# Speed and test-sized validation

Two checks that need a GPU, run once in a single session. The output of each training notebook below is attached as input, and its `best.pt` is found by run name.

1. ms per image for every detector family on the same 200 validation frames, batch 1, fp16. Timing covers the model and its NMS or post-processing; JPEG decoding and Ultralytics' CPU letterbox are excluded, except for the tiled run, whose time covers the whole frame and its crops end to end.
2. Validation on test-sized frames. No test frame is wider than 1920 px (test widths 680-1920), while 4 of the 7 validation clips are 2688-3840 px wide. Those clips are halved, boxes and areas scaled with them, and every model is scored again.

In [1]:
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, retinanet_resnet50_fpn_v2
from torchvision.models.detection.anchor_utils import AnchorGenerator
from ultralytics import RTDETR, YOLO

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load, select_frames
from src.evaluate import evaluate, to_coco
from src.predict import load_image, predict_tiled, predict_torchvision, predict_yolo

KAGGLE = Path("/kaggle/input").exists()
DATA = next(Path("/kaggle/input").rglob("train/annotations_coco.json")).parents[1] if KAGGLE else ROOT / "data/raw"
WEIGHTS = Path("/kaggle/input") if KAGGLE else ROOT / "runs"
WORK = Path("/kaggle/working") if KAGGLE else ROOT / "runs/local"
SMALL = Path("/kaggle/temp/val_small") if KAGGLE else WORK / "val_small"
DEVICE = "cuda"

RUNS = {  # best run of each family and the ensemble members; frcnn_1280_5ep has the same architecture as anchor16
    "yolo11s_1280_5ep_lr001": dict(family="YOLO11s", imgsz=1280),
    "yolo11s_1536_5ep_lr001": dict(family="YOLO11s", imgsz=1536),
    "yolo11s_1280_3ep_lr001_tile960": dict(family="YOLO11s", imgsz=1280, tile=960),
    "frcnn_1280_5ep_anchor16": dict(family="Faster R-CNN", imgsz=1280),
    "retina_1280_5ep_gamma1": dict(family="RetinaNet", imgsz=1280),
    "rtdetr-l_960_20ep_lr0002": dict(family="RT-DETR-l", imgsz=960),
}
torch.manual_seed(42)
print(torch.cuda.get_device_name(0))

Tesla T4


## Models

Ultralytics weights load as they were saved. The torchvision runs use torchvision's default proposal counts and anchors (the `anchor16` run halves every anchor size), so the constructor with 9 classes (8 + background) rebuilds them exactly.

In [2]:
def weights(run):
    # Ultralytics saves <run>/weights/best.pt, the torchvision notebook <run>/best.pt
    return next(WEIGHTS.rglob(f"{run}/weights/best.pt"), None) or next(WEIGHTS.rglob(f"{run}/best.pt"))

def load_model(run, imgsz):
    if run.startswith(("yolo", "rtdetr")):
        return (RTDETR if run.startswith("rtdetr") else YOLO)(weights(run))
    build = fasterrcnn_resnet50_fpn_v2 if run.startswith("frcnn") else retinanet_resnet50_fpn_v2
    thresholds = dict(box_score_thresh=0.001, box_detections_per_img=500) if run.startswith("frcnn") else dict(score_thresh=0.001, detections_per_img=500)
    model = build(weights=None, weights_backbone=None, num_classes=9, min_size=imgsz, max_size=imgsz, **thresholds)
    if run.endswith("anchor16"):
        model.rpn.anchor_generator = AnchorGenerator(((16,), (32,), (64,), (128,), (256,)), ((0.5, 1.0, 2.0),) * 5)
    model.load_state_dict(torch.load(weights(run), map_location=DEVICE))
    return model.to(DEVICE).eval()

def predict(model, run, images, batch):
    if "tile" in RUNS[run]:
        return predict_tiled(model, images, RUNS[run]["imgsz"], RUNS[run]["tile"])
    if run.startswith(("yolo", "rtdetr")):
        return predict_yolo(model, images, RUNS[run]["imgsz"], batch)
    return predict_torchvision(model, images, DEVICE, batch)

models = {run: load_model(run, cfg["imgsz"]) for run, cfg in RUNS.items()}
print({run: f"{sum(p.numel() for p in m.parameters()) / 1e6:.1f}M" for run, m in models.items()})

{'yolo11s_1280_5ep_lr001': '9.4M', 'yolo11s_1536_5ep_lr001': '9.4M', 'yolo11s_1280_3ep_lr001_tile960': '9.4M', 'frcnn_1280_5ep_anchor16': '43.3M', 'retina_1280_5ep_gamma1': '36.5M', 'rtdetr-l_960_20ep_lr0002': '32.8M'}


## Speed: same 200 frames, batch 1

Ultralytics reports pre-processing, inference and post-processing times per image; inference + post-processing is the comparable part. torchvision is timed around the model call, which includes its own resize and normalisation on the GPU. Ten frames warm up each model first.

In [3]:
val_coco = load(DATA / "val/annotations_coco.json")
frames = [(i["id"], DATA / "val" / i["file_name"]) for i in select_frames(val_coco)[::14][:200]]
speed = []
for run, model in models.items():
    predict(model, run, frames[:10], 1)
    if run.startswith(("yolo", "rtdetr")) and "tile" not in RUNS[run]:
        times = []
        for _, path in frames:
            r = model.predict(str(path), imgsz=RUNS[run]["imgsz"], conf=0.001, iou=0.7, max_det=500, half=True, verbose=False)[0]
            times.append(r.speed["inference"] + r.speed["postprocess"])
        ms = float(np.mean(times))
    else:
        ms = predict(model, run, frames, 1)[1]
    speed.append(dict(run_id=run, family=RUNS[run]["family"], imgsz=RUNS[run]["imgsz"],
                      params_m=sum(p.numel() for p in model.parameters()) / 1e6, ms_img=ms, fps=1000 / ms))
speed = pd.DataFrame(speed).round(1)
speed.to_csv(WORK / "speed.csv", index=False)
speed

,run_id,family,imgsz,params_m,ms_img,fps
0,yolo11s_1280_5ep_lr001,YOLO11s,1280,9.4,15.6,63.9
1,yolo11s_1536_5ep_lr001,YOLO11s,1536,9.4,15.0,66.5
2,yolo11s_1280_3ep_lr001_tile960,YOLO11s,1280,9.4,258.5,3.9
3,frcnn_1280_5ep_anchor16,Faster R-CNN,1280,43.3,69.5,14.4
4,retina_1280_5ep_gamma1,RetinaNet,1280,36.5,53.0,18.9
5,rtdetr-l_960_20ep_lr0002,RT-DETR-l,960,32.0,54.2,18.4


## Test-sized validation

Clips wider than 1920 px are halved with the same JPEG quality as the export; annotations, ignore regions and areas are scaled with them, so the size buckets follow the new pixels as they would on the test set.

In [4]:
small_coco = json.loads(json.dumps(val_coco))
scale = {i["id"]: 0.5 if i["width"] > 1920 else 1.0 for i in small_coco["images"]}
for i in small_coco["images"]:
    i["width"], i["height"] = round(i["width"] * scale[i["id"]]), round(i["height"] * scale[i["id"]])
for a in small_coco["annotations"]:
    a["bbox"] = [v * scale[a["image_id"]] for v in a["bbox"]]
    a["area"] *= scale[a["image_id"]] ** 2

def shrink(image):
    out = SMALL / image["file_name"]
    out.parent.mkdir(parents=True, exist_ok=True)
    picture = Image.open(DATA / "val" / image["file_name"])
    if scale[image["id"]] < 1:
        picture = picture.resize((image["width"], image["height"]), Image.LANCZOS)
    picture.save(out, quality=95)

with ThreadPoolExecutor(8) as pool:
    list(pool.map(shrink, small_coco["images"]))
(WORK / "val_small_coco.json").write_text(json.dumps(small_coco))
small_frames = [(i["id"], SMALL / i["file_name"]) for i in select_frames(small_coco)]
print(sum(s < 1 for s in scale.values()), "of", len(scale), "frames halved")

1560 of 2846 frames halved


In [5]:
rows = []
for run, model in models.items():
    dets, _ = predict(model, run, small_frames, 4)
    (WORK / f"val_small_{run}.json").write_text(json.dumps(to_coco(dets)))
    rows.append(dict(run_id=run, family=RUNS[run]["family"], **evaluate(small_coco, dets)))
small = pd.DataFrame(rows).round(4)
small.to_csv(WORK / "val_small.csv", index=False)
small

,run_id,family,mAP,AP50,AP_small,AP_medium,AP_large
0,yolo11s_1280_5ep_lr001,YOLO11s,0.2287,0.4354,0.1082,0.2563,0.4602
1,yolo11s_1536_5ep_lr001,YOLO11s,0.2437,0.4597,0.1162,0.2777,0.4424
2,yolo11s_1280_3ep_lr001_tile960,YOLO11s,0.2570,0.4910,0.1402,0.2872,0.5140
3,frcnn_1280_5ep_anchor16,Faster R-CNN,0.2172,0.4575,0.1064,0.2556,0.4805
4,retina_1280_5ep_gamma1,RetinaNet,0.1633,0.3136,0.0529,0.1958,0.4516
5,rtdetr-l_960_20ep_lr0002,RT-DETR-l,0.1979,0.3777,0.0853,0.2187,0.3566
